# Non-Food Cohorts Push Module

Mirrors price + cart rule changes from any source module (M2, M3, M4, M5) to a new set of **non-food cohorts** (one per existing region).

## Visibility Logic
- **Non-food SKUs** (where `categories.section_id` is in the non-food set): visible (with min-PU hide rule from `disable_pu_visibility` sheet)
- **Food SKUs**: customized but **invisible for ALL PUs** (so they don't fall back to the parent cohort and become orderable)

## Custom push method
This module has its own push method (does NOT reuse `push_prices()` or `push_cart_rules()`). It builds the upload Excel files itself with explicit `Visibility (YES/NO)` per row, then calls the low-level `post_prices()` and `post_cart_rules()` API helpers from `push_prices_handler` and `push_cart_rules_handler`.

## Usage

```python
%run non_food_cohorts_push.ipynb

# After Module 2/3/4 push (warehouse-level df with new_price/new_cart_rule):
result = push_to_non_food_cohorts(df_output, source_module='module_2', mode='live')
send_summary_to_slack(result)

# After Module 5 push (cohort-level PU-level df with price):
result = push_to_non_food_cohorts(new_intros, source_module='module_5', mode='live')
send_summary_to_slack(result)
```

In [ ]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================
import pandas as pd
import numpy as np
import os
import sys
import tempfile
from datetime import datetime
import pytz

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath('.'))

import setup_environment_2
setup_environment_2.initialize_env()
from db import query_snowflake
from common_functions import send_text_slack, send_file_slack

# Load API helpers (post_prices, post_cart_rules) and disable_pu_visibility loader
%run push_prices_handler.ipynb
%run push_cart_rules_handler.ipynb
%run queries_module.ipynb

CAIRO_TZ = pytz.timezone('Africa/Cairo')

print("Non-food cohorts push module: imports loaded")

In [ ]:
# =============================================================================
# CONSTANTS
# =============================================================================

# Section IDs that classify a product as non-food
NON_FOOD_SECTION_IDS = {714, 626, 516, 417, 351, 121, 285, 20, 54, 37, 10, 43, 44, 36, 14, 8, 55, 619, 622, 621}

# Mapping from existing cohort IDs to new non-food cohort IDs (FAKE - replace with real IDs once created)
NON_FOOD_COHORT_MAP = {
    700: 1288,    # Cairo
    701: 1289,    # Giza
    702: 1290,    # Alexandria
    703: 1291,    # Delta West
    704: 1292,    # Delta East
    1123: 1293,   # Upper Egypt - Menya
    1124: 1294,   # Upper Egypt - Assiut
    1125: 1295,   # Upper Egypt - Sohag
    1126: 1296,   # Upper Egypt - Beni Suef
}

SLACK_CHANNEL_NAME = 'new-pricing-logic'
SLACK_CHANNEL_ID = 'C0AAWK97Z3Q'

# Cart rule bounds (matches push_cart_rules_handler defaults)
MIN_CART_RULE = 2
MAX_CART_RULE = 150

print(f"Non-food sections: {len(NON_FOOD_SECTION_IDS)} | Cohort mappings: {len(NON_FOOD_COHORT_MAP)}")

In [ ]:
# =============================================================================
# SECTION LOOKUP
# =============================================================================

def get_product_sections():
    """Return product_id -> section_id mapping from products + categories."""
    q = """
    SELECT p.id AS product_id, c.section_id
    FROM products p
    JOIN categories c ON c.id = p.category_id
    """
    df = query_snowflake(q)
    df.columns = df.columns.str.lower()
    df['product_id'] = pd.to_numeric(df['product_id'], errors='coerce').astype('Int64')
    df['section_id'] = pd.to_numeric(df['section_id'], errors='coerce').astype('Int64')
    return df.dropna(subset=['product_id'])

print("get_product_sections() defined")

In [ ]:
# =============================================================================
# INPUT NORMALIZATION
# =============================================================================
# Different source modules pass different shapes:
#   - M2/M3/M4: warehouse-level rows with product_id, warehouse_id, cohort_id,
#               stocks, current_price, new_price, current_cart_rule, new_cart_rule
#               (new_price is per BASIC unit)
#   - M5:       cohort-level PU-level rows with cohort_id, product_id,
#               packing_unit_id, sku, brand, cat, price (per PU)
#
# normalize_input() produces a common shape:
#   columns = product_id, cohort_id, packing_unit_id, basic_unit_count,
#             pu_price, cart_rule
# =============================================================================

def _normalize_columns(df):
    """Lower-case columns to handle Snowflake's uppercase quirk."""
    df = df.copy()
    df.columns = [c.lower() if isinstance(c, str) else c for c in df.columns]
    return df


def _is_m5_shape(df):
    """M5 shape has packing_unit_id + price (no new_price column)."""
    cols = set(df.columns)
    return 'packing_unit_id' in cols and 'price' in cols and 'new_price' not in cols


def normalize_input(df_input, pus):
    """Normalize any source df into a common shape:
       (product_id, cohort_id, packing_unit_id, basic_unit_count, pu_price, cart_rule)
    """
    df = _normalize_columns(df_input)
    pus = _normalize_columns(pus)

    if _is_m5_shape(df):
        out = df[['cohort_id', 'product_id', 'packing_unit_id', 'price']].copy()
        out = out.rename(columns={'price': 'pu_price'})
        out = out.merge(
            pus[['product_id', 'packing_unit_id', 'basic_unit_count']],
            on=['product_id', 'packing_unit_id'], how='left'
        )
        out['basic_unit_count'] = pd.to_numeric(out['basic_unit_count'], errors='coerce').fillna(1)
        out['cart_rule'] = np.nan
        out['pu_price'] = pd.to_numeric(out['pu_price'], errors='coerce')
        return out.dropna(subset=['pu_price'])

    # M2/M3/M4 shape: warehouse-level basic-unit prices
    required = ['product_id', 'cohort_id', 'new_price']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' missing from input df")

    df['stocks'] = pd.to_numeric(df.get('stocks', 1), errors='coerce').fillna(1)
    df['new_price'] = pd.to_numeric(df['new_price'], errors='coerce')

    # Filter to rows with a new price
    df = df.dropna(subset=['new_price'])
    if 'current_price' in df.columns:
        df['current_price'] = pd.to_numeric(df['current_price'], errors='coerce')
        df = df[df['new_price'] != df['current_price']]

    if df.empty:
        return pd.DataFrame(columns=['product_id', 'cohort_id', 'packing_unit_id',
                                      'basic_unit_count', 'pu_price', 'cart_rule'])

    # Stock-weighted average price per (product_id, cohort_id)
    grp = df.groupby(['product_id', 'cohort_id'], as_index=False)
    def _wavg(g):
        w = g['stocks'].clip(lower=0)
        wsum = w.sum()
        if wsum > 0:
            return (g['new_price'] * w).sum() / wsum
        return g['new_price'].mean()
    basic = grp.apply(lambda g: pd.Series({'basic_price': _wavg(g)})).reset_index()
    basic = basic[['product_id', 'cohort_id', 'basic_price']]

    # Cart rule = min across warehouses (matches push_cart_rules_handler)
    if 'new_cart_rule' in df.columns:
        df['new_cart_rule'] = pd.to_numeric(df['new_cart_rule'], errors='coerce')
        cart = df.dropna(subset=['new_cart_rule']).groupby(
            ['product_id', 'cohort_id'], as_index=False
        )['new_cart_rule'].min().rename(columns={'new_cart_rule': 'cart_rule'})
        basic = basic.merge(cart, on=['product_id', 'cohort_id'], how='left')
    else:
        basic['cart_rule'] = np.nan

    # Expand to all PUs of each product
    expanded = basic.merge(
        pus[['product_id', 'packing_unit_id', 'basic_unit_count']],
        on='product_id', how='inner'
    )
    expanded['basic_unit_count'] = pd.to_numeric(expanded['basic_unit_count'], errors='coerce').fillna(1)
    expanded['pu_price'] = (expanded['basic_price'] * expanded['basic_unit_count']).round(2)
    return expanded[['product_id', 'cohort_id', 'packing_unit_id',
                     'basic_unit_count', 'pu_price', 'cart_rule']]


print("normalize_input() defined")

In [ ]:
# =============================================================================
# CUSTOM PUSH METHOD
# =============================================================================
# Builds Excel upload files per cohort with explicit Visibility column,
# then calls post_prices() and post_cart_rules() (low-level API helpers
# from push_prices_handler / push_cart_rules_handler) per cohort.
# =============================================================================

def _set_visibility(df_norm):
    """Set Visibility (YES/NO) per row based on is_non_food + min-PU disable rule."""
    df_norm = df_norm.sort_values(['cohort_id', 'product_id', 'basic_unit_count']).copy()
    df_norm['min_pu_flag'] = df_norm.groupby(['cohort_id', 'product_id'])['basic_unit_count'].rank(method='first').eq(1).astype(int)

    # Load disable_pu_visibility list (same source as push_prices_handler)
    try:
        disable_min = get_disable_pu_visibility()
        disable_set = set(pd.to_numeric(disable_min['product_id'], errors='coerce').dropna().astype(int).unique())
    except Exception as e:
        print(f"Warning: could not load disable_pu_visibility ({e}); proceeding with no min-PU hide")
        disable_set = set()

    def _vis(row):
        # Food: invisible for ALL PUs
        if not row['is_non_food']:
            return 'NO'
        # Non-food + smallest PU + product flagged for hide
        if row['min_pu_flag'] == 1 and int(row['product_id']) in disable_set:
            return 'NO'
        # Hardcoded exception (same as push_prices_handler)
        if int(row['product_id']) == 1309 and int(row['packing_unit_id']) == 23:
            return 'NO'
        return 'YES'

    df_norm['Visibility'] = df_norm.apply(_vis, axis=1)
    return df_norm


def _build_price_excel(group, file_path):
    """Write a price upload Excel for one cohort."""
    payload = pd.DataFrame({
        'Product ID': group['product_id'].astype(int),
        'Product Name': group.get('sku', pd.Series([''] * len(group))).fillna(''),
        'Packing Unit ID': group['packing_unit_id'].astype(int),
        'Price': group['pu_price'].astype(float).round(2),
        'Visibility (YES/NO)': group['Visibility'],
    })
    payload.to_excel(file_path, index=False)


def _build_cart_excel(group, file_path):
    """Write a cart rule upload Excel for one cohort."""
    payload = pd.DataFrame({
        'Product ID': group['product_id'].astype(int),
        'Packing Unit ID': group['packing_unit_id'].astype(int),
        'Cart Rules': group['cart_rule_clean'].astype(int),
    })
    payload.to_excel(file_path, index=False)


def custom_push(df_norm, source_module, mode):
    """Push prices + cart rules per cohort. Builds Excels, calls API helpers.

    Args:
        df_norm: normalized df with columns
                 product_id, cohort_id, packing_unit_id, basic_unit_count,
                 pu_price, cart_rule, is_non_food (+ optional 'sku')
        source_module: tag for filenames / logging
        mode: 'testing' (build files only, no upload) or 'live' (upload to API)

    Returns:
        dict with pushed_cohorts, failed_cohorts, file_paths, rows
    """
    if df_norm.empty:
        return {'pushed_cohorts': [], 'failed_cohorts': [], 'file_paths': [], 'rows': 0}

    df_norm = _set_visibility(df_norm)

    # Cart rule cleanup: convert larger PUs from basic units to packing units, clip to bounds
    df_norm['cart_rule_clean'] = df_norm.apply(
        lambda r: max(MIN_CART_RULE, min(MAX_CART_RULE, int(round((r['cart_rule'] or 0) / max(r['basic_unit_count'], 1)))))
        if pd.notna(r['cart_rule']) and r['cart_rule'] > 0
        else None,
        axis=1
    )

    pushed_prices, failed_prices = [], []
    pushed_carts, failed_carts = [], []
    file_paths = []

    out_dir = tempfile.gettempdir()
    timestamp = datetime.now(CAIRO_TZ).strftime('%Y%m%d_%H%M%S')

    for cohort_id, group in df_norm.groupby('cohort_id'):
        cohort_int = int(cohort_id)
        # ---- Prices ----
        price_file = os.path.join(out_dir, f'nonfood_prices_{source_module}_{cohort_int}_{timestamp}.xlsx')
        _build_price_excel(group, price_file)
        file_paths.append(price_file)

        if mode == 'live':
            try:
                resp = post_prices(cohort_int, price_file)
                if resp.status_code == 200 and b'"success":true' in resp.content:
                    pushed_prices.append(cohort_int)
                else:
                    print(f"  Cohort {cohort_int} prices: status {resp.status_code} | {resp.content[:200]}")
                    failed_prices.append(cohort_int)
            except Exception as e:
                print(f"  Cohort {cohort_int} prices error: {e}")
                failed_prices.append(cohort_int)
        else:
            pushed_prices.append(cohort_int)
            print(f"  [TESTING] Cohort {cohort_int} prices: {price_file} ({len(group)} rows)")

        # ---- Cart rules (only push rows with a cart rule value) ----
        cart_group = group.dropna(subset=['cart_rule_clean'])
        if cart_group.empty:
            continue

        cart_file = os.path.join(out_dir, f'nonfood_carts_{source_module}_{cohort_int}_{timestamp}.xlsx')
        _build_cart_excel(cart_group, cart_file)
        file_paths.append(cart_file)

        if mode == 'live':
            try:
                resp = post_cart_rules(cohort_int, cart_file)
                if resp.status_code == 200 and b'Cart Rules Updated Successfully' in resp.content:
                    pushed_carts.append(cohort_int)
                else:
                    print(f"  Cohort {cohort_int} cart: status {resp.status_code} | {resp.content[:200]}")
                    failed_carts.append(cohort_int)
            except Exception as e:
                print(f"  Cohort {cohort_int} cart error: {e}")
                failed_carts.append(cohort_int)
        else:
            pushed_carts.append(cohort_int)
            print(f"  [TESTING] Cohort {cohort_int} cart: {cart_file} ({len(cart_group)} rows)")

    return {
        'pushed_prices_cohorts': pushed_prices,
        'failed_prices_cohorts': failed_prices,
        'pushed_carts_cohorts': pushed_carts,
        'failed_carts_cohorts': failed_carts,
        'file_paths': file_paths,
        'rows': len(df_norm),
    }


print("custom_push() defined")

In [ ]:
# =============================================================================
# MAIN ENTRY POINT
# =============================================================================

def push_to_non_food_cohorts(df_input, source_module='unknown', mode='testing'):
    """
    Mirror price + cart rule changes from a source module to the non-food cohorts.

    Args:
        df_input: df from M2/M3/M4 (warehouse-level basic-unit prices) or
                  M5 (cohort-level PU-level prices)
        source_module: tag for filenames / Slack
        mode: 'testing' (build files only) or 'live' (upload to API)

    Returns:
        dict with rows pushed, food/non-food counts, pushed/failed cohorts
    """
    print(f"\n{'='*60}")
    print(f"NON-FOOD COHORTS PUSH | source: {source_module} | mode: {mode}")
    print(f"{'='*60}")

    if df_input is None or len(df_input) == 0:
        print("No input rows. Skipping.")
        return {'source_module': source_module, 'rows': 0,
                'food_count': 0, 'non_food_count': 0,
                'pushed_prices_cohorts': [], 'failed_prices_cohorts': [],
                'pushed_carts_cohorts': [], 'failed_carts_cohorts': []}

    # Step 1: load packing units lookup
    pus = get_packing_units()

    # Step 2: normalize input shape
    df_norm = normalize_input(df_input, pus)
    if df_norm.empty:
        print("Normalized df is empty (no price changes to mirror). Skipping.")
        return {'source_module': source_module, 'rows': 0,
                'food_count': 0, 'non_food_count': 0,
                'pushed_prices_cohorts': [], 'failed_prices_cohorts': [],
                'pushed_carts_cohorts': [], 'failed_carts_cohorts': []}

    # Step 3: keep only rows in source cohorts we have a mapping for
    df_norm = df_norm[df_norm['cohort_id'].isin(NON_FOOD_COHORT_MAP.keys())].copy()
    if df_norm.empty:
        print(f"No rows in source cohorts {list(NON_FOOD_COHORT_MAP.keys())}. Skipping.")
        return {'source_module': source_module, 'rows': 0,
                'food_count': 0, 'non_food_count': 0,
                'pushed_prices_cohorts': [], 'failed_prices_cohorts': [],
                'pushed_carts_cohorts': [], 'failed_carts_cohorts': []}

    # Step 4: remap cohort_id to the non-food cohort IDs
    df_norm['source_cohort_id'] = df_norm['cohort_id']
    df_norm['cohort_id'] = df_norm['cohort_id'].map(NON_FOOD_COHORT_MAP)

    # Step 5: classify food vs non-food
    sections = get_product_sections()
    df_norm = df_norm.merge(sections, on='product_id', how='left')
    df_norm['is_non_food'] = df_norm['section_id'].apply(
        lambda s: pd.notna(s) and int(s) in NON_FOOD_SECTION_IDS
    )

    food_count = int((~df_norm['is_non_food']).sum())
    non_food_count = int(df_norm['is_non_food'].sum())
    print(f"  Total rows: {len(df_norm)}")
    print(f"  Non-food (visible): {non_food_count} rows")
    print(f"  Food (customized invisible): {food_count} rows")
    print(f"  Cohorts: {sorted(df_norm['cohort_id'].unique().tolist())}")

    # Step 6: push (build files + upload)
    push_result = custom_push(df_norm, source_module, mode)

    result = {
        'source_module': source_module,
        'mode': mode,
        'rows': push_result['rows'],
        'food_count': food_count,
        'non_food_count': non_food_count,
        'pushed_prices_cohorts': push_result['pushed_prices_cohorts'],
        'failed_prices_cohorts': push_result['failed_prices_cohorts'],
        'pushed_carts_cohorts': push_result['pushed_carts_cohorts'],
        'failed_carts_cohorts': push_result['failed_carts_cohorts'],
        'file_paths': push_result['file_paths'],
    }

    print(f"\n{'='*60}")
    print(f"DONE | prices pushed: {len(result['pushed_prices_cohorts'])}, failed: {len(result['failed_prices_cohorts'])}")
    print(f"     | carts  pushed: {len(result['pushed_carts_cohorts'])}, failed: {len(result['failed_carts_cohorts'])}")
    print(f"{'='*60}\n")
    return result


print("push_to_non_food_cohorts() defined")

In [ ]:
# =============================================================================
# SLACK NOTIFICATION
# =============================================================================

def send_summary_to_slack(result):
    """Send a Slack summary of the push result."""
    timestamp = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M')
    failed = (len(result.get('failed_prices_cohorts', [])) +
              len(result.get('failed_carts_cohorts', [])))
    status = 'Completed' if failed == 0 else 'Completed with failures'

    msg = (
        f"*Non-Food Cohorts Push: {status}*\n"
        f"\n"
        f"*Source:* `{result.get('source_module', 'unknown')}`\n"
        f"*Mode:* `{result.get('mode', 'unknown')}`\n"
        f"*Time:* {timestamp} Cairo\n"
        f"\n"
        f"*Rows:* {result.get('rows', 0)}\n"
        f"*Non-food (visible):* {result.get('non_food_count', 0)}\n"
        f"*Food (customized invisible):* {result.get('food_count', 0)}\n"
        f"\n"
        f"*Prices:* pushed {len(result.get('pushed_prices_cohorts', []))}, "
        f"failed {len(result.get('failed_prices_cohorts', []))}\n"
        f"*Cart rules:* pushed {len(result.get('pushed_carts_cohorts', []))}, "
        f"failed {len(result.get('failed_carts_cohorts', []))}\n"
    )
    if result.get('failed_prices_cohorts') or result.get('failed_carts_cohorts'):
        msg += (
            f"\n*Failed price cohorts:* {result.get('failed_prices_cohorts', [])}\n"
            f"*Failed cart cohorts:* {result.get('failed_carts_cohorts', [])}\n"
        )

    try:
        send_text_slack(SLACK_CHANNEL_NAME, msg)
        print("Slack summary sent")
    except Exception as e:
        print(f"Failed to send Slack summary: {e}")


print("send_summary_to_slack() defined")
print("\nModule ready. Use:")
print("  result = push_to_non_food_cohorts(df, source_module='module_X', mode='testing')")
print("  send_summary_to_slack(result)")